---
## 1. Installazione dipendenze

In [ ]:
!pip install -q huggingface_hub datasets pandas tqdm

---
## 2. Configurazione e autenticazione

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cartella di destinazione su Google Drive
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/SHIELD/Dataset/'

import os
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print(f'✅ Google Drive montato. Cartella di output: {DRIVE_OUTPUT_DIR}')

In [ ]:
import os
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Token caricato dai Colab Secrets")
except Exception:
    HF_TOKEN = input("Inserisci il tuo Hugging Face token: ")
    print("✅ Token inserito manualmente")

os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
MODEL_ID = "Qwen/Qwen3-235B-A22B-Instruct-2507"
DATASET_ID = ""

SYSTEM_PROMPT = """You are a professional medical translator specialized in radiology. Your task is to translate chest radiology reports from English into Italian. This is not a word-for-word translation: you must adapt sentence structure and phrasing to match standard Italian radiology report conventions, while preserving the exact clinical meaning. Use strictly standard Italian radiology report language as used in clinical PACS systems, with impersonal and formal constructions, and avoid conversational forms. Do not add, remove, or interpret information. Maintain a neutral, objective, and technical radiological report style. Keep abbreviations unchanged unless a standard Italian equivalent is commonly used, and apply it consistently. Do not summarize, simplify, explain, or comment on the content. Output only the translated report text in Italian."""

print(f"Modello: {MODEL_ID}")
print(f"Dataset: {DATASET_ID}")
print(f"System prompt: {len(SYSTEM_PROMPT)} caratteri")

---
## 3. Caricamento dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import hashlib

print("Caricamento dataset...")
dataset = load_dataset(DATASET_ID, split="train")
print(f"✅ Dataset caricato: {len(dataset)} righe")
print(f"Colonne: {dataset.column_names}")

df = dataset.to_pandas()

def image_to_md5(image_dict):
    if isinstance(image_dict, dict) and 'bytes' in image_dict:
        return hashlib.md5(image_dict['bytes']).hexdigest()
    return None

df['image'] = df['image'].apply(image_to_md5)

print(f"✅ Colonna image convertita in MD5")
print(f"\nEsempio ID: {df['image'].iloc[0]}")
df.head()

---
## 4. Funzione di traduzione


In [ ]:
import time
import threading
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from huggingface_hub import InferenceClient
from tqdm.notebook import tqdm

# Codici HTTP che indicano errori non recuperabili (non ha senso ritentare)
NON_RECOVERABLE_CODES = ["401", "402", "403", "404"]

# Ogni thread usa il proprio client per evitare conflitti
def make_client():
    return InferenceClient(
        model=MODEL_ID,
        token=HF_TOKEN,
    )

# Thread-local storage: ogni thread ha il proprio InferenceClient
_thread_local = threading.local()

def get_client():
    if not hasattr(_thread_local, "client"):
        _thread_local.client = make_client()
    return _thread_local.client


def translate_text(text: str, max_retries: int = 30, initial_wait: float = 5.0) -> str:
    """
    Traduce un singolo testo dall'inglese all'italiano utilizzando il modello Qwen3
    tramite HF Inference API.

    Args:
        text: Testo in inglese da tradurre.
        max_retries: Numero massimo di tentativi in caso di errore.
        initial_wait: Tempo di attesa fisso (secondi) tra i tentativi.

    Returns:
        Testo tradotto in italiano, oppure stringa vuota in caso di errore persistente.
    """
    # Salta testi vuoti o NaN
    if not text or not isinstance(text, str) or text.strip() == "":
        return ""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text.strip()},
    ]

    client = get_client()

    for attempt in range(max_retries):
        try:
            response = client.chat_completion(
                messages=messages,
                max_tokens=2048,
                temperature=0.7,
                top_p = 0.8
            )
            content = response.choices[0].message.content

            return content.strip()

        except Exception as e:
            error_str = str(e)

            # Errori non recuperabili: esci subito senza ritentare
            if any(code in error_str for code in NON_RECOVERABLE_CODES):
                print(f"  ❌ Errore non recuperabile (tentativo {attempt + 1}): {error_str[:200]}")
                return ""

            wait_time = initial_wait * (2 ** min(attempt, 6))
            if attempt < max_retries - 1:
                print(f"  ⚠️ Tentativo {attempt + 1}/{max_retries} fallito: {e}")
                print(f"  ⏳ Attesa {wait_time:.0f}s prima del prossimo tentativo...")
                time.sleep(wait_time)
            else:
                print(f"  ❌ Tutti i {max_retries} tentativi falliti per il testo: {text[:80]}...")
                return ""


# Test rapido
test_text = "No acute cardiopulmonary abnormality."
print(f"Test input:  {test_text}")
test_output = translate_text(test_text)
print(f"Test output: {test_output}")

---
## OpenRouter Version

In [ ]:
from openai import OpenAI
import threading
import time

OPENROUTER_API_KEY = ""
MODEL_ID = "qwen/qwen3-235b-a22b-2507"

# Codici HTTP che indicano errori non recuperabili
NON_RECOVERABLE_CODES = ["401", "402", "403", "404"]

_thread_local = threading.local()

def get_client():
    if not hasattr(_thread_local, "client"):
        _thread_local.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=OPENROUTER_API_KEY,
        )
    return _thread_local.client


def translate_text(text: str, max_retries: int = 30, initial_wait: float = 5.0) -> str:
    """
    Traduce un singolo testo dall'inglese all'italiano utilizzando il modello Qwen3
    tramite OpenRouter API.

    Args:
        text: Testo in inglese da tradurre.
        max_retries: Numero massimo di tentativi in caso di errore.
        initial_wait: Tempo di attesa fisso (secondi) tra i tentativi.

    Returns:
        Testo tradotto in italiano, oppure stringa vuota in caso di errore persistente.
    """
    if not text or not isinstance(text, str) or text.strip() == "":
        return ""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text.strip()},
    ]

    client = get_client()

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=messages,
                max_tokens=2048,
                temperature=0.7,
                top_p=0.8,
            )
            content = response.choices[0].message.content

            return content.strip()

        except Exception as e:
            error_str = str(e)

            # Errori non recuperabili: esci subito senza ritentare
            if any(code in error_str for code in NON_RECOVERABLE_CODES):
                print(f"  ❌ Errore non recuperabile (tentativo {attempt + 1}): {error_str[:200]}")
                return ""

            wait_time = initial_wait * (2 ** min(attempt, 6))
            if attempt < max_retries - 1:
                print(f"  ⚠️ Tentativo {attempt + 1}/{max_retries} fallito: {e}")
                print(f"  ⏳ Attesa {wait_time:.0f}s prima del prossimo tentativo...")
                time.sleep(wait_time)
            else:
                print(f"  ❌ Tutti i {max_retries} tentativi falliti per il testo: {text[:80]}...")
                return ""

In [ ]:
def translate_dataframe_parallel(
    df_input: pd.DataFrame,
    columns_to_translate: list,
    save_path: str = None,
    save_every: int = 50,
    n_workers: int = 1, # Da modificare se ci sono limiti sul numero di richieste al minuto
) -> pd.DataFrame:
    """
    Traduce le colonne specificate di un DataFrame in parallelo, aggiungendo colonne _it.

    Args:
        df_input: DataFrame di input.
        columns_to_translate: Lista di colonne da tradurre (es. ["findings", "impression"]).
        save_path: Percorso dove salvare checkpoint intermedi (opzionale).
        save_every: Ogni quante righe completate salvare un checkpoint.
        n_workers: Numero di thread paralleli per le chiamate API.

    Returns:
        DataFrame con le nuove colonne tradotte.
    """
    df_out = df_input.copy()

    # Inizializza le colonne di output se non esistono
    for col in columns_to_translate:
        col_it = f"{col}_it"
        if col_it not in df_out.columns:
            df_out[col_it] = ""

    total_rows = len(df_out)

    # Pre-copia i testi sorgente per thread-safety
    # Evita letture concorrenti sul DataFrame durante la traduzione
    source_data = {}
    for col in columns_to_translate:
        source_data[col] = {
            idx: str(df_out.at[df_out.index[idx], col]) if pd.notna(df_out.at[df_out.index[idx], col]) else ""
            for idx in range(total_rows)
        }

    # Identifica le righe da tradurre (salta quelle già completate per tutte le colonne)
    def row_needs_translation(idx):
        for col in columns_to_translate:
            val_en = df_out.at[df_out.index[idx], col]
            val_it = df_out.at[df_out.index[idx], f"{col}_it"]

            has_en = pd.notna(val_en) and str(val_en).strip() != ""
            has_it = pd.notna(val_it) and str(val_it).strip() != ""

            if has_en and not has_it:
                return True
        return False

    todo_indices = [i for i in range(total_rows) if row_needs_translation(i)]
    n_todo = len(todo_indices)
    n_done_already = total_rows - n_todo

    print(f"\n{'='*60}")
    print(f"  Traduzione PARALLELA ({n_workers} worker)")
    print(f"  Righe totali:       {total_rows}")
    print(f"  Già tradotte:       {n_done_already}")
    print(f"  Da tradurre:        {n_todo}")
    print(f"  Colonne:            {columns_to_translate}")
    print(f"{'='*60}\n")

    if n_todo == 0:
        print("✅ Tutte le righe sono già tradotte!")
        return df_out

    # Lock per accesso thread-safe al DataFrame e al contatore
    lock = threading.Lock()
    completed_count = [0]  # lista per mutabilità da closure
    start_time = time.time()

    def translate_row(idx):
        """Traduce tutte le colonne di una singola riga."""
        results = {}
        for col in columns_to_translate:
            col_it = f"{col}_it"
            # Ricontrolla col lock se è già tradotta (race condition)
            with lock:
                existing = df_out.at[df_out.index[idx], col_it]
                val_en = df_out.at[df_out.index[idx], col]

            has_en = pd.notna(val_en) and str(val_en).strip() != ""
            has_it = pd.notna(existing) and str(existing).strip() != ""

            if has_it or not has_en:
                continue

            # Usa i dati pre-copiati (thread-safe, nessun accesso al DataFrame)
            text = source_data[col][idx]
            translation = translate_text(text)
            results[col_it] = translation

        # Scrivi i risultati nel DataFrame con lock
        with lock:
            for col_it, translation in results.items():
                df_out.at[df_out.index[idx], col_it] = translation

            completed_count[0] += 1
            current = completed_count[0]

            # Checkpoint periodico
            if save_path and current % save_every == 0:
                # Backup del file esistente prima di sovrascrivere
                if os.path.exists(save_path):
                    shutil.copy2(save_path, save_path + ".bak")
                df_out.to_csv(save_path, index=False)
                elapsed = time.time() - start_time
                speed = current / elapsed * 60  # righe/minuto
                remaining = (n_todo - current) / speed if speed > 0 else 0
                print(f"  💾 Checkpoint ({current}/{n_todo}) — "
                      f"{speed:.1f} righe/min — "
                      f"~{remaining:.0f} min rimanenti")

        return idx

    # Esecuzione parallela
    with tqdm(total=n_todo, desc="⚡ Traduzione parallela", unit="riga") as pbar:
        with ThreadPoolExecutor(max_workers=n_workers) as executor:
            futures = {executor.submit(translate_row, idx): idx for idx in todo_indices}
            for future in as_completed(futures):
                try:
                    future.result()
                except Exception as e:
                    print(f"  ❌ Errore riga {futures[future]}: {e}")
                pbar.update(1)

    # Salvataggio finale
    if save_path:
        if os.path.exists(save_path):
            shutil.copy2(save_path, save_path + ".bak")
        df_out.to_csv(save_path, index=False)
        print(f"\n✅ Risultato finale salvato in: {save_path}")

    elapsed = time.time() - start_time
    print(f"\n⏱️ Tempo totale: {elapsed/60:.1f} minuti")
    print(f"📊 Velocità media: {n_todo / elapsed * 60:.1f} righe/minuto")

    return df_out

---
## 100 Sample

In [ ]:
# Seleziona i primi 100 sample
df_100 = df.head(10).copy()
print(f"Primi 100 sample selezionati.")
print(f"Colonne disponibili: {list(df_100.columns)}")

In [ ]:
df_100_translated = translate_dataframe_parallel(
    df_input=df_100,
    columns_to_translate=["findings", "impression"],
    save_path="mimic_cxr_100_translated.csv",
    save_every=25,
)

print(f"\n✅ Traduzione dei primi 100 sample completata!")

In [ ]:
print("=" * 80)
print("ESEMPI DI TRADUZIONE (primi 100 sample)")
print("=" * 80)

for i in range(min(5, len(df_100_translated))):
    print(f"\n--- Sample {i+1} ---")

    findings_en = df_100_translated.iloc[i].get("findings", "")
    findings_it = df_100_translated.iloc[i].get("findings_it", "")
    impression_en = df_100_translated.iloc[i].get("impression", "")
    impression_it = df_100_translated.iloc[i].get("impression_it", "")

    print(f"\n📋 FINDINGS (EN):")
    print(f"  {str(findings_en)[:300]}")
    print(f"📋 FINDINGS (IT):")
    print(f"  {str(findings_it)[:300]}")

    print(f"\n📋 IMPRESSION (EN):")
    print(f"  {str(impression_en)[:300]}")
    print(f"📋 IMPRESSION (IT):")
    print(f"  {str(impression_it)[:300]}")

print("\n" + "=" * 80)

In [ ]:
# Statistiche sui primi 100 sample
n_findings_ok = df_100_translated["findings_it"].apply(lambda x: bool(str(x).strip())).sum()
n_impression_ok = df_100_translated["impression_it"].apply(lambda x: bool(str(x).strip())).sum()
n_findings_tot = df_100_translated["findings"].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False).sum()
n_impression_tot = df_100_translated["impression"].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False).sum()

print(f"📊 Statistiche traduzione (primi 100):")
print(f"   Findings tradotti:   {n_findings_ok}/{n_findings_tot}")
print(f"   Impression tradotti: {n_impression_ok}/{n_impression_tot}")

In [ ]:
# Download del file CSV (primi 100)
try:
    from google.colab import files
    files.download("mimic_cxr_100_translated.csv")
    print("📥 Download avviato per mimic_cxr_100_translated.csv")
except ImportError:
    print("ℹ️ Non in Google Colab. Il file è salvato come mimic_cxr_100_translated.csv")

---
## Traduzione dataset

In [ ]:
CHECKPOINT_EVERY = 50  # Salva su Drive ogni N righe completate

FULL_SAVE_PATH = f"{DRIVE_OUTPUT_DIR}/mimic_cxr_full_translated.csv"

if os.path.exists(FULL_SAVE_PATH):
    print(f"🔄 Checkpoint trovato su Drive: {FULL_SAVE_PATH}")
    try:
        df_full = pd.read_csv(FULL_SAVE_PATH)
        print(f"   ✅ CSV caricato correttamente")
    except Exception as e:
        print(f"   ⚠️ CSV corrotto: {e}")
        bak_path = FULL_SAVE_PATH + ".bak"
        if os.path.exists(bak_path):
            print(f"   🔄 Tentativo di recupero dal backup: {bak_path}")
            try:
                df_full = pd.read_csv(bak_path)
                print(f"   ✅ Backup caricato correttamente")
            except Exception as e2:
                print(f"   ❌ Anche il backup è corrotto: {e2}")
                print(f"   🆕 Ripartendo da zero...")
                df_full = df.copy()
        else:
            print(f"   ❌ Nessun backup disponibile")
            print(f"   🆕 Ripartendo da zero...")
            df_full = df.copy()

    for col in ["findings", "impression"]:
        col_it = f"{col}_it"
        if col_it not in df_full.columns:
            df_full[col_it] = ""
    df_full[["findings_it", "impression_it"]] = df_full[["findings_it", "impression_it"]].fillna("")

    # Conta righe già tradotte (entrambe le colonne devono essere compilate)
    def is_col_done(col_name):
        has_orig = df_full[col_name].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False)
        has_trad = df_full[f"{col_name}_it"].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False)
        return ~has_orig | has_trad

    already_done = (is_col_done("findings") & is_col_done("impression")).sum()
    print(f"   ✅ Righe già tradotte (entrambe le colonne): {already_done}/{len(df_full)}")
    print(f"   ▶️  Riprendo dalla riga {already_done}...")
else:
    print(f"🆕 Nessun checkpoint trovato. Inizio traduzione da zero.")
    df_full = df.copy()
    print(f"   Dataset completo: {len(df_full)} righe")

print(f"💾 Checkpoint ogni: {CHECKPOINT_EVERY} righe")
print(f"📁 File di output: {FULL_SAVE_PATH}")

In [ ]:
df_full_translated = translate_dataframe_parallel(
    df_input=df_full,
    columns_to_translate=["findings", "impression"],
    save_path=FULL_SAVE_PATH,
    save_every=CHECKPOINT_EVERY
)

print(f"\n🎉 Traduzione del dataset completo terminata!")
print(f"   File salvato su Google Drive: {FULL_SAVE_PATH}")

In [ ]:
# 1. Ricarica il dataset originale con le immagini
print("Caricamento dataset originale...")
dataset = load_dataset(DATASET_ID, split="train")
df_original = dataset.to_pandas()
print(f"✅ Dataset originale caricato: {len(df_original)} righe")

# 2. Calcola md5 in una colonna separata, lasciando 'image' intatta
df_original['image_md5'] = df_original['image'].apply(image_to_md5)

# 3. Carica il CSV tradotto
df_translated = pd.read_csv(FULL_SAVE_PATH)
print(f"✅ CSV tradotto caricato: {len(df_translated)} righe")

# 4. Verifica unicità MD5 prima del merge
n_unique_original = df_original['image_md5'].nunique()
n_unique_translated = df_translated['image'].nunique()
print(f"\n🔍 Verifica unicità MD5:")
print(f"   Originale:  {n_unique_original} unici su {len(df_original)} righe")
print(f"   Tradotto:   {n_unique_translated} unici su {len(df_translated)} righe")

assert n_unique_original == len(df_original), (
    f"MD5 non univoci nel dataset originale! {n_unique_original} unici su {len(df_original)} righe"
)
assert n_unique_translated == len(df_translated), (
    f"MD5 non univoci nel CSV tradotto! {n_unique_translated} unici su {len(df_translated)} righe. "
    f"Rimuovere i duplicati con: df_translated = df_translated.drop_duplicates(subset='image', keep='last')"
)
print("   ✅ MD5 univoci — merge sicuro")

# 5. Merge su md5
df_final = df_original.merge(
    df_translated[["image", "findings_it", "impression_it"]].rename(columns={"image": "image_md5"}),
    on="image_md5",
    how="left"
)
print(f"\n✅ Merge completato: {len(df_final)} righe")

# 6. Verifica che il merge non abbia prodotto duplicati
assert len(df_final) == len(df_original), (
    f"Il merge ha prodotto duplicati! {len(df_final)} righe vs {len(df_original)} originali"
)

# 7. Verifica integrità immagini tramite md5
print("\n🔍 Verifica integrità immagini...")
df_final['image_md5_check'] = df_final['image'].apply(image_to_md5)
integrity_ok = (df_final['image_md5'] == df_final['image_md5_check']).all()
n_mismatch = (df_final['image_md5'] != df_final['image_md5_check']).sum()

if integrity_ok:
    print(f"   ✅ Tutte le {len(df_final)} immagini sono integre")
else:
    print(f"   ❌ {n_mismatch} immagini con md5 non corrispondente!")
    print(df_final[df_final['image_md5'] != df_final['image_md5_check']][['image_md5', 'image_md5_check']])

df_final = df_final.drop(columns=["image_md5", "image_md5_check"])

# 8. Verifica traduzioni
missing_findings = df_final["findings_it"].isna().sum()
missing_impression = df_final["impression_it"].isna().sum()
print(f"\n📊 Controllo qualità traduzioni:")
print(f"   findings_it mancanti:   {missing_findings}")
print(f"   impression_it mancanti: {missing_impression}")

# 9. Salva come Parquet
PARQUET_SAVE_PATH = f"{DRIVE_OUTPUT_DIR}/mimic_cxr_final.parquet"
df_final.to_parquet(PARQUET_SAVE_PATH, index=False)
print(f"\n✅ Dataset finale salvato in: {PARQUET_SAVE_PATH}")
print(f"   Colonne: {list(df_final.columns)}")
df_final.head()

In [ ]:
# Statistiche finali sul dataset completo
n_findings_ok = df_full_translated["findings_it"].apply(lambda x: bool(str(x).strip())).sum()
n_impression_ok = df_full_translated["impression_it"].apply(lambda x: bool(str(x).strip())).sum()
n_findings_tot = df_full_translated["findings"].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False).sum()
n_impression_tot = df_full_translated["impression"].apply(lambda x: bool(str(x).strip()) if pd.notna(x) else False).sum()

print(f"\n📊 Statistiche traduzione (dataset completo):")
print(f"   Findings tradotti:   {n_findings_ok}/{n_findings_tot}")
print(f"   Impression tradotti: {n_impression_ok}/{n_impression_tot}")
print(f"   Errori findings:     {n_findings_tot - n_findings_ok}")
print(f"   Errori impression:   {n_impression_tot - n_impression_ok}")

In [ ]:
# Visualizza alcuni risultati dal dataset completo
import random

print("=" * 80)
print("ESEMPI DI TRADUZIONE (campione casuale dal dataset completo)")
print("=" * 80)

sample_indices = random.sample(range(len(df_full_translated)), min(5, len(df_full_translated)))

for i in sample_indices:
    print(f"\n--- Sample {i} ---")

    findings_en = df_full_translated.iloc[i].get("findings", "")
    findings_it = df_full_translated.iloc[i].get("findings_it", "")
    impression_en = df_full_translated.iloc[i].get("impression", "")
    impression_it = df_full_translated.iloc[i].get("impression_it", "")

    print(f"\n📋 FINDINGS (EN):")
    print(f"  {str(findings_en)[:300]}")
    print(f"📋 FINDINGS (IT):")
    print(f"  {str(findings_it)[:300]}")

    print(f"\n📋 IMPRESSION (EN):")
    print(f"  {str(impression_en)[:300]}")
    print(f"📋 IMPRESSION (IT):")
    print(f"  {str(impression_it)[:300]}")

print("\n" + "=" * 80)

In [ ]:
# Il dataset completo è già salvato su Google Drive
import os
file_size_mb = os.path.getsize(FULL_SAVE_PATH) / (1024 * 1024)
print(f"📁 Il dataset tradotto è salvato su Google Drive:")
print(f"   {FULL_SAVE_PATH}")
print(f"   Dimensione: {file_size_mb:.1f} MB")
print(f"\nPuoi accedere al file direttamente dal tuo Google Drive ")
print(f"nella cartella: traduzione_radiologia/")